In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = "agent-book"


# LLM/Chat Model

## Completion API(legacy)

In [2]:
from langchain_openai import OpenAI
model = OpenAI(model="gpt-5.4-nano", temperature=0)
output = model.invoke("반갑습니다")
print(output)

. 2024년 1월 1일 기준으로, 한국의 공휴일은 다음과 같습니다. 다만, 공휴일은 매년 변경될 수 있으므로, 정확한 확인을 위해서는 고용노동부 또는 관공서 공휴일 안내를 참고하시기 바랍니다.

2024년 한국 공휴일(대체공휴일 포함)은 다음과 같습니다.

1월
- 1월 1일(월): 신정
- 1월 10일(수): 대체공휴일(설날 연휴 중 해당일이 주말인 경우)

2월
- 2월 9일(금): 설날
- 2월 10일(토): 설날
- 2월 11일(일): 설날
- 2월 12일(월): 대체공휴일(설날 연휴 중 해당일이 주말인 경우)

3월
- 3월 1일(금): 삼일절

4월
- 4월 10일(수): 국회의원 선거일(해당 시)
- 4월 15일(월): 4·15 총선


## Chat Model(Chat Completions API)

In [3]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-5.4-nano", temperature=0)

messages = [
    SystemMessage("You are a helpful assistant"),
    HumanMessage("안녕하세요. 저는 희수라고합니다"),
    AIMessage(content="안녕하세요, 희수님!. 어떤 도움이 필요하신가요?"),
    HumanMessage(content="제 이름을 아시나요?")
]

ai_message = model.invoke(messages)
print(ai_message.content)

네, 희수님이라고 말씀해주셨어요. 😊


## 스트리밍

In [4]:
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-5.4-nano", temperature=0)
messages = [
    SystemMessage("You are a helpful assistant."),
    HumanMessage("안녕하세요!")
]

for chunk in model.stream(messages):
    print(chunk.content, end="", flush=True)

안녕하세요! 😊  
무엇을 도와드릴까요?

# Prompt Template
## PromptTemplate

In [5]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """
    다음 요리의 레시피를 생각해 주세요.

    요리명: {dish}
    """
)

prompt_value = prompt.invoke({"dish": "카레"})
print(prompt_value.text)


    다음 요리의 레시피를 생각해 주세요.

    요리명: 카레
    


## ChatPromptTemplate

In [6]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "사용자가 입력한 요리의 레시피를 생각해주세요"),
        ("human", "{dish}")
    ]
)

prompt_value = prompt.invoke({"dish": "카레"})
print(prompt_value)

messages=[SystemMessage(content='사용자가 입력한 요리의 레시피를 생각해주세요', additional_kwargs={}, response_metadata={}), HumanMessage(content='카레', additional_kwargs={}, response_metadata={})]


## MessagesPlaceholder

In [7]:
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        MessagesPlaceholder("chat_history", optional=True),
        ("human", "{input}")
    ]
)

prompt_value = prompt.invoke(
    {
        "chat_history": [
            HumanMessage(content="안녕하ㅔ요! 저는 희수라고 합니다!"),
            AIMessage("안녕하세요, 희수님! 어떻게 도와드릴까요?")
        ],
        "input": "제 이름을 아시나요?"
    }
)
print(prompt_value)

messages=[SystemMessage(content='You are a helpful assistant.', additional_kwargs={}, response_metadata={}), HumanMessage(content='안녕하ㅔ요! 저는 희수라고 합니다!', additional_kwargs={}, response_metadata={}), AIMessage(content='안녕하세요, 희수님! 어떻게 도와드릴까요?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='제 이름을 아시나요?', additional_kwargs={}, response_metadata={})]


# Output Parser
## PydanticOutputParser

In [8]:
from pydantic import BaseModel, Field

class Recipe(BaseModel):
    ingredients: list[str] = Field(description="ingredients of the dish")
    steps: list[str] = Field(description="steps to make the dish")

from langchain_core.output_parsers import PydanticOutputParser

output_parser = PydanticOutputParser(pydantic_object=Recipe)

format_instructions = output_parser.get_format_instructions()
print(format_instructions)

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"ingredients": {"description": "ingredients of the dish", "items": {"type": "string"}, "title": "Ingredients", "type": "array"}, "steps": {"description": "steps to make the dish", "items": {"type": "string"}, "title": "Steps", "type": "array"}}, "required": ["ingredients", "steps"]}
```


In [9]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        ("system",
        "사용자가 입력한 요리의 레시피를 생각해주세요.\n\n"
        "{format_instructions}"),
        ("human", "{dish}")
        
    ]
)

prompt_with_format_instructions = prompt.partial(
    format_instructions=format_instructions
)

prompt_value = prompt_with_format_instructions.invoke({"dish": "카레"})
print("=== role: system === ")
print(prompt_value.messages[0].content)
print("=== role: user === ")
print(prompt_value.messages[1].content)

=== role: system === 
사용자가 입력한 요리의 레시피를 생각해주세요.

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"ingredients": {"description": "ingredients of the dish", "items": {"type": "string"}, "title": "Ingredients", "type": "array"}, "steps": {"description": "steps to make the dish", "items": {"type": "string"}, "title": "Steps", "type": "array"}}, "required": ["ingredients", "steps"]}
```
=== role: user === 
카레


In [10]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-5.4-nano", temperature=0)

ai_message = model.invoke(prompt_value)
print(ai_message.content)

{
  "ingredients": [
    "양파 1개(중간크기), 다짐",
    "당근 1/2개, 다짐",
    "감자 1개, 깍둑썰기",
    "카레 고형/가루 1박스(또는 1/2~1팩, 취향에 따라)",
    "식용유 2~3큰술",
    "소고기(또는 돼지고기/닭고기) 300g(선택)",
    "다진 마늘 1큰술(선택)",
    "생강 약간(선택)",
    "물 700~800ml(카레 농도에 따라 조절)",
    "소금 약간(선택)",
    "후추 약간(선택)",
    "밥(곁들임)"
  ],
  "steps": [
    "(고기 사용 시) 고기를 한입 크기로 썰어 소금과 후추로 가볍게 밑간합니다.",
    "냄비에 식용유를 두르고 중불에서 양파를 5~7분 충분히 볶아 갈색이 나게(단맛을 내기) 만듭니다.",
    "(선택) 다진 마늘과 생강을 넣고 30초~1분 더 볶습니다.",
    "고기를 넣고 겉면이 익을 때까지 3~5분 볶습니다.",
    "당근과 감자를 넣고 2~3분 더 볶습니다.",
    "물(700~800ml)을 붓고 센 불에서 끓인 뒤, 중약불로 줄여 감자가 익을 때까지 15~25분 끓입니다.",
    "카레 고형을 넣고 잘 풀어가며 저어줍니다. 농도가 맞도록 5~10분 더 끓입니다.",
    "간을 보고 소금이나 후추로 마무리합니다(원하면 생략 가능).",
    "따뜻한 밥에 얹어 완성합니다."
  ]
}


In [11]:
recipe = output_parser.invoke(ai_message)
print(type(recipe))
print(recipe)

<class '__main__.Recipe'>
ingredients=['양파 1개(중간크기), 다짐', '당근 1/2개, 다짐', '감자 1개, 깍둑썰기', '카레 고형/가루 1박스(또는 1/2~1팩, 취향에 따라)', '식용유 2~3큰술', '소고기(또는 돼지고기/닭고기) 300g(선택)', '다진 마늘 1큰술(선택)', '생강 약간(선택)', '물 700~800ml(카레 농도에 따라 조절)', '소금 약간(선택)', '후추 약간(선택)', '밥(곁들임)'] steps=['(고기 사용 시) 고기를 한입 크기로 썰어 소금과 후추로 가볍게 밑간합니다.', '냄비에 식용유를 두르고 중불에서 양파를 5~7분 충분히 볶아 갈색이 나게(단맛을 내기) 만듭니다.', '(선택) 다진 마늘과 생강을 넣고 30초~1분 더 볶습니다.', '고기를 넣고 겉면이 익을 때까지 3~5분 볶습니다.', '당근과 감자를 넣고 2~3분 더 볶습니다.', '물(700~800ml)을 붓고 센 불에서 끓인 뒤, 중약불로 줄여 감자가 익을 때까지 15~25분 끓입니다.', '카레 고형을 넣고 잘 풀어가며 저어줍니다. 농도가 맞도록 5~10분 더 끓입니다.', '간을 보고 소금이나 후추로 마무리합니다(원하면 생략 가능).', '따뜻한 밥에 얹어 완성합니다.']


## StrOutputParser


In [12]:
from langchain_core.messages import AIMessage
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

ai_message = AIMessage(content="안녕하세요. 저는 AI 어시스턴트입니다.")
ai_message = output_parser.invoke(ai_message)
print(type(ai_message))
print(ai_message)

<class 'langchain_core.messages.base.TextAccessor'>
안녕하세요. 저는 AI 어시스턴트입니다.


# Chain - LangChain Expression Language(LCEL) 개요
## prompt와 model의 연결

In [13]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "사용자가 입력한 요리의 레시피를 생각해주세요."),
        ("human", "{dish}")
    ]
)

model = ChatOpenAI(model_name="gpt-5.4-nano", temperature=0)


# '|'로 chain 생성
chain = prompt | model

# 실행
ai_message = chain.invoke({"dish": "카레"})
print(ai_message.content)

좋아요! **카레 레시피** 하나 추천해드릴게요. (기본 가정용, 맛 보장형)

## 1) 준비 재료 (4인분)
- 양파 1~2개 (중간 크기, 큼직하게)
- 감자 2개
- 당근 1개
- 쇠고기(또는 돼지고기) 300g (없으면 생략 가능)
- 카레 가루 4~6큰술 (또는 카레 루 1/2~1봉)
- 식용유 2~3큰술
- 물 800~1000ml
- 소금 약간 (필요시)
- (선택) 사과 1/4개 또는 꿀 1작은술 (단맛으로 맛↑)

## 2) 만드는 법
1. **재료 손질**
   - 양파: 채썰거나 다지기
   - 감자, 당근: 한입 크기로 썰기
   - 고기: 한입 크기로 썰기

2. **고기 볶기(선택이지만 추천)**
   - 냄비에 식용유를 두르고 고기를 볶아 겉면을 살짝 익혀요.
   - (고기가 없으면 다음 단계로 바로 가셔도 됩니다.)

3. **양파 볶기(핵심)**
   - 양파를 넣고 **중불~약불에서 15~25분** 충분히 볶아 갈색이 나올 때까지 졸여주세요.
   - 여기서 맛이 크게 좌우돼요.

4. **채소 넣고 끓이기**
   - 감자, 당근을 넣고 1~2분 더 볶은 뒤,
   - 물(800~1000ml)을 넣고 끓여요.
   - 감자가 익을 때까지 **중불로 15~20분**.

5. **카레 만들기(카레 가루/루에 따라)**
   - **카레 루(덩어리) 사용 시:**  
     약불로 줄이고 카레를 넣어가며 저어 **완전히 녹이기**. (5~10분)
   - **카레 가루 사용 시:**  
     카레 가루를 물에 조금씩 풀어 넣거나, 덩어리가 안 생기게 저어 넣고 **5~8분** 끓여 농도 맞추기.

6. **간 맞추기**
   - 맛을 보고 소금으로 간을 살짝 조절하거나,
   - 너무 진하면 물을 조금, 너무 묽으면 약불로 더 졸여요.

## 3) 완성 팁
- **가장 맛있는 포인트는 양파를 오래 볶는 것**이에요.
- 더 진하고 부드럽게 원하면: 마지막에 **우유 2~4큰술** 또는 **버터 1~2큰술**을 넣어도 좋아요(

In [14]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | model | StrOutputParser()

output = chain.invoke({"dish": "카레"})
print(output)

좋아요! **기본 카레 레시피(4인분)**로 소개해드릴게요.  

## 재료
- 양파 1개(중간, 얇게 채썰기)
- 감자 2개(중간, 깍둑썰기)
- 당근 1개(깍둑썰기)
- 닭고기 또는 쇠고기 300~400g(한입 크기)
- 카레가루 80~100g *(시판 고형 카레면 1~2블록 또는 포장 기준대로)*
- 물 700~900ml
- 식용유 2~3큰술
- (선택) 다진 마늘 1큰술, 생강 약간
- (선택) 우스터소스 1~2큰술(풍미)
- 소금 약간(필요시)

## 만드는 법
1. **재료 손질**
   - 감자/당근은 깍둑썰고, 양파는 얇게 썰어요.
   - 고기는 한입 크기로 썰어 소금 한 꼬집(선택) 정도만 해둡니다.

2. **양파 볶기(핵심)**
   - 냄비에 식용유를 두르고 중불~약불로 양파를 볶아요.
   - 양파가 **갈색이 날 정도로 충분히**(약 10~15분) 달달하고 진해질 때까지 볶으면 맛이 확 좋아져요.

3. **고기 볶기**
   - 고기를 넣고 겉면이 살짝 익을 때까지 3~5분 볶습니다.
   - (선택) 다진 마늘/생강도 이때 같이 넣어 향을 내요.

4. **채소 넣고 끓이기**
   - 감자와 당근을 넣고 2~3분 더 볶은 뒤,
   - 물(700~900ml)을 부어 끓입니다.
   - 끓기 시작하면 중불로 **감자가 부드러워질 때까지** 약 15~20분 끓여요.

5. **카레 넣기**
   - 불을 줄인 뒤 카레가루(또는 고형 카레)를 넣습니다.
   - 덩어리가 생기지 않게 저어주고,
   - 걸쭉함이 원하는 정도가 될 때까지 3~8분 더 끓여요.
   - (선택) 우스터소스를 넣으면 더 깊은 맛이 납니다.

6. **간 맞추기**
   - 맛을 보고 필요하면 소금으로 살짝 조절한 뒤 불을 끄고 마무리!

## 맛있게 하는 팁
- **양파를 오래 볶는 것**이 카레 맛의 좌우 요인이에요.
- 카레가 너무 되면 물을 조금 더, 너무 묽으면 2~3분 더 끓이세요.

원하시는 스타일이 있어요?  
예) **돼지고기/소고기/닭고기*

## PydanticOutputParser를 사용한 연결

In [15]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

class Recipe(BaseModel):
    ingredients : list[str] = Field(description="ingredients of the dish")
    steps: list[str] = Field(description="steps to make the dish")

output_parser = PydanticOutputParser(pydantic_object=Recipe)

In [16]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_messages([
    ("system", "사용자가 입력한 요리의 레시피를 생각해주세요.\n\n{format_instructions}"),
    ("human", "{dish}")
])

prompt_with_format_instructions = prompt.partial(
    format_instructions=output_parser.get_format_instructions()
)

model = ChatOpenAI(model="gpt-5.4-nano", temperature=0).bind(response_format={"type":"json_object"})

chain = prompt_with_format_instructions | model | output_parser

recipe = chain.invoke({"dish" : "카레"})
print(type(recipe))
print(recipe)


<class '__main__.Recipe'>
ingredients=['식용유 또는 카놀라유 2~3큰술', '양파 1개(중간 크기), 채썰기', '당근 1/2~1개, 깍둑썰기', '감자 1개(중간 크기), 깍둑썰기', '돼지고기(또는 닭고기/소고기) 200~300g, 한입 크기', '카레 가루(또는 카레 블록) 100~140g(제품 지침 기준)', '물 800ml~1L(카레 농도에 맞게)', '우유 또는 생크림(선택) 2~3큰술(부드러움용)', '소금 약간(간 조절)', '후추 약간', '다진 마늘 1큰술(선택)', '버터 1큰술(선택, 풍미용)'] steps=['재료 준비: 양파는 채썰고, 당근·감자는 깍둑썰기 합니다. 고기는 한입 크기로 썰어 소금/후추로 가볍게 밑간합니다.', '팬에 기름을 두르고 중불로 양파를 볶습니다. 양파가 갈색이 되고 달달한 향이 나올 때까지 8~15분 정도 충분히 볶아 주세요. (선택: 버터 1큰술 추가)', '고기를 넣고 3~5분 볶아 겉면을 익힙니다. (선택: 다진 마늘을 함께 볶아 향을 냅니다.)', '당근과 감자를 넣고 2~3분 더 볶습니다.', '물(800ml~1L)을 붓고 끓입니다. 끓어오르면 중불로 줄여 감자가 익을 때까지 15~25분 정도 조리합니다. (감자 크기에 따라 시간 조절)', '카레 가루를 쓰는 경우: 불을 약하게 줄이고 카레를 조금씩 풀어가며 저어 농도를 맞춥니다. (덩어리 생기지 않게 충분히 저어주세요) 카레 블록을 쓰는 경우: 카레 블록을 넣고 완전히 녹여 저어가며 끓입니다.', '10분 정도 더 끓여 맛을 내고, 농도와 간을 확인합니다. 필요하면 소금으로 간을 맞춥니다.', '(선택) 마지막에 우유/생크림을 2~3큰술 넣고 한 번 더 저어 부드럽게 만든 뒤 불을 끕니다.', '밥에 올려 뜨겁게 서빙합니다.']


## with_structured_output

In [17]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

class Recipe(BaseModel):
    ingredients : list[str] = Field(description="ingredients of the dish")
    steps: list[str] = Field(description="steps to make the dish")


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "사용자가 입력한 요리의 레시피를 생각해주세요"),
        ("human", "{dish}")
    ]
)

model = ChatOpenAI(model="gpt-5.4-nano")
chain = prompt | model.with_structured_output(Recipe)

recipe = chain.invoke({"dish": "알리오 올리오"})
print(type(recipe))
print(recipe)

<class '__main__.Recipe'>
ingredients=['스파게티(면) 200g', '엑스트라버진 올리브오일 4~6큰술', '마늘 6~8쪽(얇게 슬라이스)', '청양고추 또는 페페론치노 1~2개(선택)', '소금(면수용)', '후추 약간', '파슬리 다진 것(또는 파슬리 가루) 1~2큰술', '(선택) 레몬즙 또는 레몬 제스트 약간', '(선택) 파마산 치즈(서빙용) 약간'] steps=['큰 냄비에 물을 넉넉히 끓이고 소금을 충분히 넣어(면수 간) 스파게티를 포장지 시간보다 1분 정도 덜 삶아주세요.', '마늘은 얇게 슬라이스하고, 페페론치노/청양고추는 잘게 썰어 준비합니다.', '팬에 올리브오일을 두르고 약불~중약불에서 마늘을 볶습니다. 마늘이 노릇해지기 시작하면 바로 불을 끄거나 약하게 줄여 타지 않게 조절하세요.', '(선택) 페페론치노/청양고추를 넣고 10~20초만 향을 내줍니다.', '소스가 될 수 있도록 스파게티 면수를 국자로 1~2국자 덜어두고, 면을 건져 팬에 넣습니다.', '면을 마늘오일과 잘 섞으며 면수(필요한 만큼)를 조금씩 넣어 농도를 맞춥니다. 30초~1분 정도 빠르게 버무리듯 조리해요.', '후추를 뿌리고(선택) 레몬즙 몇 방울 또는 레몬 제스트를 추가합니다.', '불을 끄고 파슬리를 뿌린 뒤 즉시 맛을 보고 소금 간을 조절해 서빙합니다. (선택: 파마산 치즈 추가)']


# LangChain의 RAG 관련 컴포넌트

## Document Loader
- 데이터 읽기에 사용됨

In [18]:
from langchain_community.document_loaders import GitLoader

def file_filter(file_path: str)->bool:
    return file_path.endswith(".md")

loader = GitLoader(
    clone_url="https://github.com/langchain-ai/langchain",
    repo_path="./langchain",
    branch="master",
    file_filter=file_filter
)

raw_docs = loader.load()

print(type(raw_docs))
print(len(raw_docs)) # 읽어들인 데이터의 건수



<class 'list'>
28


## Document Transformer

In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
docs = text_splitter.split_documents(raw_docs)
print(len(docs))

95


## Embedding Model
- 텍스트 벡터화

In [20]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

query = "AWS의 S3에서 데이터를 읽어들이기 위한 Document Loader가 있나요?"
vector = embeddings.embed_query(query)
print(len(vector))
print(vector)

1536
[0.006885528564453125, 0.007720947265625, 0.0214996337890625, -0.0269012451171875, 0.035125732421875, 0.00441741943359375, -0.005336761474609375, 0.0228729248046875, 0.0110626220703125, -0.022186279296875, -0.00626373291015625, -0.0008625984191894531, -0.00957489013671875, -0.0035247802734375, -0.014404296875, 0.04705810546875, 0.004276275634765625, -0.0193023681640625, -0.0251007080078125, 0.00537109375, 0.0228729248046875, 0.034820556640625, -0.02679443359375, 0.04766845703125, 0.0302276611328125, -0.048919677734375, -0.0223388671875, 0.0282135009765625, -0.0175018310546875, -0.10736083984375, -0.01544952392578125, -0.05621337890625, -0.0252227783203125, 0.03472900390625, -0.0157318115234375, 0.048431396484375, 0.05010986328125, 0.012786865234375, -0.006290435791015625, -0.025726318359375, 0.005672454833984375, -0.00423431396484375, 0.0195159912109375, 0.0029239654541015625, 0.0245513916015625, 0.025115966796875, -0.0036067962646484375, 0.013946533203125, -0.05224609375, 0.00177

## Vector Store

In [21]:
from langchain_chroma import Chroma
db = Chroma.from_documents(docs, embeddings) # vector store 초기화

In [22]:
retriever = db.as_retriever()

query = "AWS의 S3에서 데이터를 읽어들이기 위한 Document Loader가 있나요?"
context_docs = retriever.invoke(query)

print(f"len = {len(context_docs)}")
first_doc = context_docs[0]
print(f"metadata = {first_doc.metadata}")
print(first_doc.page_content)

len = 4
metadata = {'source': 'libs/partners/exa/README.md', 'file_type': '.md', 'file_path': 'libs/partners/exa/README.md', 'file_name': 'README.md'}
View the [documentation](https://docs.langchain.com/oss/python/integrations/providers/exa_search) for more details.


## LCEL을 사용한 RAG Chain 구현

In [23]:
from langchain_core.prompts import ChatMessagePromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

prompt = ChatPromptTemplate.from_template("""
                                          다음 문맥만을 바탕으로 질문에 답변해주세요.
                                          문맥: ''' {context} '''
                                          질문:  {question}
""")

model = ChatOpenAI(model="gpt-5.4-nano", temperature=0)
chain = (
    {"context": retriever, "question": RunnablePassthrough()} | 
    prompt |
    model |
    StrOutputParser()
)

output = chain.invoke(query)
print(output)

주어진 문맥에는 **AWS S3에서 데이터를 읽어들이기 위한 Document Loader**가 있는지에 대한 정보가 없습니다. 문맥에는 LangChain의 “Text Splitters”와 일부 “partners(통합)” 구조만 제시되어 있을 뿐, S3 관련 로더(예: S3DocumentLoader 등) 내용은 확인되지 않습니다.
